In [9]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# remove NaN values
df = pd.read_csv('gw_data.csv')
columns = ['mass_1_source','mass_2_source','total_mass_source','final_mass_source','luminosity_distance','network_matched_filter_snr','chi_eff','redshift']
df_new = df.dropna(subset= columns).copy()
print('Remaining: ',len(df_new))

#Define new parameters
df_new['chirp_mass_det'] = df_new['chirp_mass_source'] * (1 + df_new['redshift'])
df_new['q'] = df_new['mass_1_source'] / df_new['mass_2_source']
df_new['eta'] = df_new['q'] / (1 + df_new['q'])**2
df_new['f_rad'] = (df_new['total_mass_source'] - df_new['final_mass_source']) / df_new['total_mass_source']
df_new['chi_eff_sqr'] = (df_new['chi_eff'])**2

#Take logarithms
df_new['log_snr'] = np.log10(df_new['network_matched_filter_snr'])
df_new['log_mchirp_det'] = np.log10(df_new['chirp_mass_det'])
df_new['log_mchirp_src'] = np.log10(df_new['chirp_mass_source'])
df_new['log_final_mass'] = np.log10(df_new['final_mass_source'])
df_new['log_dl'] = np.log10(df_new['luminosity_distance'])
df_new['log_eta'] = np.log10(df_new['eta'])
df_new['log_q'] = np.log10(df_new['q'])
df_new['log_1_z'] = np.log10(1 + df_new['redshift'])
df_new = df_new[df_new['f_rad'] > 0].copy()
df_new['log_frad'] = np.log10(df_new['f_rad'])

Remaining:  269


In [5]:
# 1. DETECTION / SELECTION CONTROL 
x1 = df_new[['log_mchirp_det','log_dl']]
x1 = sm.add_constant(x1)
y1 = df_new['log_snr']

model1 = sm.OLS(y1,x1).fit()
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:                log_snr   R-squared:                       0.601
Model:                            OLS   Adj. R-squared:                  0.598
Method:                 Least Squares   F-statistic:                     200.6
Date:                Fri, 14 Aug 2026   Prob (F-statistic):           7.49e-54
Time:                        09:37:59   Log-Likelihood:                 254.74
No. Observations:                 269   AIC:                            -503.5
Df Residuals:                     266   BIC:                            -492.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              2.4415      0.070     35.

In [6]:
# 2. INTRINSIC GR / RADIATED-ENERGY PLANE
x2 = df_new[['log_eta','chi_eff']]
x2 = sm.add_constant(x2)
y2 = df_new['log_frad']

model2 = sm.OLS(y2,x2).fit()
print(model2.summary())


                            OLS Regression Results                            
Dep. Variable:               log_frad   R-squared:                       0.909
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1321.
Date:                Fri, 14 Aug 2026   Prob (F-statistic):          6.78e-139
Time:                        09:45:36   Log-Likelihood:                 521.15
No. Observations:                 269   AIC:                            -1036.
Df Residuals:                     266   BIC:                            -1026.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.4465      0.019    -23.224      0.0

In [11]:
# 3. REMNANT MASS PLANE
x3 = df_new[['log_mchirp_src','chi_eff']]
x3 = sm.add_constant(x3)
y3 = df_new['log_final_mass']
model3 = sm.OLS(y3,x3).fit()
print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:         log_final_mass   R-squared:                       0.973
Model:                            OLS   Adj. R-squared:                  0.973
Method:                 Least Squares   F-statistic:                     4758.
Date:                Fri, 14 Aug 2026   Prob (F-statistic):          6.09e-209
Time:                        10:46:53   Log-Likelihood:                 459.17
No. Observations:                 269   AIC:                            -912.3
Df Residuals:                     266   BIC:                            -901.6
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.4238      0.013     32.

In [12]:
# 4. POPULATION / SPIN-WIDTH PLANE
x4 = df_new[['log_q','log_1_z']]
x4 = sm.add_constant(x4)
y4 = df_new['chi_eff_sqr']
model4 = sm.OLS(y4,x4).fit()
print(model4.summary())

                            OLS Regression Results                            
Dep. Variable:            chi_eff_sqr   R-squared:                       0.096
Model:                            OLS   Adj. R-squared:                  0.089
Method:                 Least Squares   F-statistic:                     14.15
Date:                Fri, 14 Aug 2026   Prob (F-statistic):           1.44e-06
Time:                        11:27:28   Log-Likelihood:                 441.04
No. Observations:                 269   AIC:                            -876.1
Df Residuals:                     266   BIC:                            -865.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0162      0.008     -2.000      0.0

In [13]:
# 5. MASS-SPIN POPULATION PLANE
x5 = df_new[['log_q','log_mchirp_src']]
x5 = sm.add_constant(x5)
y5 = df_new['chi_eff_sqr']
model5 = sm.OLS(y5,x5).fit()
print(model5.summary())

                            OLS Regression Results                            
Dep. Variable:            chi_eff_sqr   R-squared:                       0.086
Model:                            OLS   Adj. R-squared:                  0.079
Method:                 Least Squares   F-statistic:                     12.46
Date:                Fri, 14 Aug 2026   Prob (F-statistic):           6.69e-06
Time:                        13:23:35   Log-Likelihood:                 439.49
No. Observations:                 269   AIC:                            -873.0
Df Residuals:                     266   BIC:                            -862.2
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.0422      0.017     -2.

In [15]:
# 6. RESIDUAL SEARCH
x6 = df_new[['log_eta','chi_eff']]
x6 = sm.add_constant(x6)
y6 = df_new['log_frad']
model6 = sm.OLS(y6,x6).fit()

delta_rad = model6.resid

x7 = df_new[['log_mchirp_det','log_1_z','log_snr','log_dl']]
x7 = sm.add_constant(x7)
y7 = delta_rad
model7= sm.OLS(y7,x7).fit()
print(model7.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     1.654
Date:                Fri, 14 Aug 2026   Prob (F-statistic):              0.161
Time:                        14:32:18   Log-Likelihood:                 524.48
No. Observations:                 269   AIC:                            -1039.
Df Residuals:                     264   BIC:                            -1021.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.0678      0.091     -0.

In [19]:
import itertools

# =====================================================================
# BLIND SEARCH (KÖR TARAMA) ALGORİTMASI
# =====================================================================

# 1. PARAMETRE HAVUZU VE GRUPLANDIRMA (Sizin sütun isimlerinize göre)
var_groups = {
    'log_mchirp_src': 'mass_src', 
    'log_final_mass': 'mass_src',  # Sizin belirlediğiniz isim
    'log_mchirp_det': 'mass_det',
    'log_q': 'ratio', 
    'log_eta': 'ratio',
    'chi_eff': 'spin', 
    'chi_eff_sqr': 'spin',         # Sizin belirlediğiniz isim
    'log_frad': 'energy',
    'log_dl': 'extrinsic', 
    'log_snr': 'extrinsic', 
    'log_1_z': 'extrinsic'
}

variables = list(var_groups.keys())
results = []

# 2. KOMBİNASYON ÜRETİCİSİ (Tüm olası 3'lü kombinasyonlar: Y, X1, X2)
for combo in itertools.permutations(variables, 3):
    Y_var = combo[0]
    X1_var = combo[1]
    X2_var = combo[2]
    
    Y_group = var_groups[Y_var]
    X1_group = var_groups[X1_var]
    X2_group = var_groups[X2_var]
    
    model_groups = [Y_group, X1_group, X2_group]
    
    # =================================================================
    # 3. ASTROFİZİKSEL DIŞLAMA KURALLARI (Sahte keşifleri engelle)
    # =================================================================
    
    # Kural 1: Aynı gruptan iki değişken yan yana gelemez (Örn: log_q ve log_eta)
    if model_groups.count('ratio') > 1: continue
    if model_groups.count('spin') > 1: continue
    if model_groups.count('mass_src') > 1: continue
        
    # Kural 2: Kozmolojik çerçeve karışımı yasak (M_det ile M_src aynı denkleme giremez)
    if 'mass_det' in model_groups and 'mass_src' in model_groups: continue
        
    # =================================================================
    # 4. ETİKETLEME (Bu denklem neyi ifade ediyor?)
    # =================================================================
    tags = []
    if 'extrinsic' in model_groups: 
        tags.append("Selection/Cosmology")
    if not ('extrinsic' in model_groups or 'mass_det' in model_groups): 
        tags.append("Intrinsic GR/Pop")
        
    # =================================================================
    # 5. MODELİN EĞİTİLMESİ (OLS)
    # =================================================================
    
    # Sizin oluşturduğunuz df_new'i kullanıyoruz
    temp_df = df_new[[Y_var, X1_var, X2_var]].copy()
    
    # Matematiksel hataları önlemek için sonsuz değerleri (inf) temizle
    temp_df = temp_df.replace([np.inf, -np.inf], np.nan).dropna()
    
    # İstatistiksel güç için en az 50 veri noktası şartı
    if len(temp_df) < 50: continue 
        
    Y = temp_df[Y_var]
    X = temp_df[[X1_var, X2_var]]
    X = sm.add_constant(X)
    
    try:
        model = sm.OLS(Y, X).fit()
        
        # En az bir katsayı istatistiksel olarak anlamlıysa (p < 0.05) listeye ekle
        if model.pvalues[X1_var] < 0.05 or model.pvalues[X2_var] < 0.05:
            results.append({
                'Y (Hedef)': Y_var,
                'X1': X1_var,
                'X2': X2_var,
                'R_squared': model.rsquared,
                'Adj_R_squared': model.rsquared_adj,
                'AIC': model.aic,
                'X1_pvalue': model.pvalues[X1_var],
                'X2_pvalue': model.pvalues[X2_var],
                'Kategori': " + ".join(tags)
            })
    except:
        continue # Olası matris tersine çevirme hatalarını (Singular matrix) es geç

# =====================================================================
# 6. SONUÇLARIN SIRALANMASI VE EKRANA YAZDIRILMASI
# =====================================================================

if len(results) > 0:
    results_df = pd.DataFrame(results)
    
    # BİLİMSEL KURAL: Modeller R-kare'ye göre DEĞİL, en düşük AIC skoruna göre sıralanır.
    # (AIC ne kadar düşükse, model o kadar sade ve bilgi vericidir)
    results_df = results_df.sort_values(by='AIC', ascending=True).reset_index(drop=True)
    
    print("--- KÖR TARAMA (BLIND SEARCH) SONUCU: EN İYİ 10 ADAY DÜZLEM ---")
    
    # Jupyter Notebook kullanıyorsanız tabloyu güzel formatta görmek için:
    try:
        display(results_df.head(10))
    except NameError:
        print(results_df.head(10).to_string())
else:
    print("Kurallara uyan istatistiksel olarak anlamlı bir model bulunamadı.")


# Sadece 'Intrinsic GR/Pop' etiketine sahip olan modelleri göster
saf_fizik_modelleri = results_df[results_df['Kategori'] == 'Intrinsic GR/Pop'].head(10)
print(saf_fizik_modelleri.to_string())

--- KÖR TARAMA (BLIND SEARCH) SONUCU: EN İYİ 10 ADAY DÜZLEM ---


,Y (Hedef),X1,X2,R_squared,Adj_R_squared,AIC,X1_pvalue,X2_pvalue,Kategori
0,log_1_z,chi_eff_sqr,log_dl,0.944663,0.944247,-1425.022391,1.846231e-04,4.336883e-167,Selection/Cosmology
1,log_1_z,log_dl,chi_eff_sqr,0.944663,0.944247,-1425.022391,4.336883e-167,1.846231e-04,Selection/Cosmology
2,log_1_z,log_dl,log_q,0.944341,0.943923,-1423.460434,1.715543e-168,4.202830e-04,Selection/Cosmology
3,log_1_z,log_q,log_dl,0.944341,0.943923,-1423.460434,4.202830e-04,1.715543e-168,Selection/Cosmology
4,log_1_z,log_dl,log_eta,0.944138,0.943718,-1422.482194,1.208620e-167,7.056665e-04,Selection/Cosmology
5,log_1_z,log_eta,log_dl,0.944138,0.943718,-1422.482194,7.056665e-04,1.208620e-167,Selection/Cosmology
6,log_1_z,chi_eff,log_dl,0.943912,0.943490,-1421.394447,1.259708e-03,8.939117e-167,Selection/Cosmology
7,log_1_z,log_dl,chi_eff,0.943912,0.943490,-1421.394447,8.939117e-167,1.259708e-03,Selection/Cosmology
8,log_1_z,log_dl,log_frad,0.942823,0.942393,-1416.222947,3.155415e-166,2.136975e-02,Selection/Cosmology
9,log_1_z,log_frad,log_dl,0.942823,0.942393,-1416.222947,2.136975e-02,3.155415e-166,Selection/Cosmology


   Y (Hedef)              X1              X2  R_squared  Adj_R_squared          AIC      X1_pvalue      X2_pvalue          Kategori
18   log_eta         chi_eff        log_frad   0.900216       0.899466 -1272.983648   6.129036e-48  7.550101e-135  Intrinsic GR/Pop
19   log_eta        log_frad         chi_eff   0.900216       0.899466 -1272.983648  7.550101e-135   6.129036e-48  Intrinsic GR/Pop
20   log_eta        log_frad     chi_eff_sqr   0.861230       0.860187 -1184.265059  6.987415e-115   8.319793e-29  Intrinsic GR/Pop
21   log_eta     chi_eff_sqr        log_frad   0.861230       0.860187 -1184.265059   8.319793e-29  6.987415e-115  Intrinsic GR/Pop
24   log_eta        log_frad  log_mchirp_src   0.785520       0.783907 -1067.142045   6.147597e-86   3.718435e-03  Intrinsic GR/Pop
25   log_eta  log_mchirp_src        log_frad   0.785520       0.783907 -1067.142045   3.718435e-03   6.147597e-86  Intrinsic GR/Pop
28   log_eta  log_final_mass        log_frad   0.780764       0.779115 -1061